# Stroke Prediction & Explainable AI — The Beginner-to-Advanced Practical Workbook
### Based on *Fundamentals of XAI, Part 3* · corrected, extended and explained cell by cell

> **Who this is for:** Learners who have never used Python, a Jupyter notebook, machine learning, or explainable AI. Start at the top and run **one code cell at a time**. By the end, you will be able to explain why each operation is performed, critique the resulting model, and reproduce the workflow on other tabular datasets.

**Inputs:** This notebook and the accompanying `healthcare-dataset-stroke-data.csv`. **Output:** A reproducible exploratory analysis, two classes of interpretable model, fair(er) out-of-sample evaluation, and global/local explanations.

**Scope:** The teaching sequence follows **PDF 3, *Fundamentals of XAI*** — pp. 3–9 imbalance/histograms, pp. 10–14 InterpretML, pp. 15–30 stroke-data laboratory. PDF 2 explains interpretable vs black-box and global vs local explanations. PDF 1 gives the motivation and limitations of XAI. Material added beyond the slides (PR-AUC, leakage protection, validation discussion, calibration and audits) is clearly labeled as an extension.

**Important safety/validity note:** This small, observational educational dataset does **not** establish a validated clinical screening tool. A model-estimated probability is not a medical diagnosis or an individual’s real-world stroke probability. XAI describes what the model learned; it does not prove causation, fairness, or clinical correctness.

**Recommended study routine:** For each step, (1) read the *Why/How*, (2) predict what the code will do, (3) run it, (4) compare the result with *Reading the result*, (5) answer any checkpoint questions. Run the notebook from the beginning after restarting the kernel.

## Orientation — how a Jupyter notebook works

A notebook is a document made of two kinds of cells:

- **Markdown cells** (like this one) teach concepts, give instructions and interpret findings. They do not run Python.
- **Code cells** execute Python instructions and often display text, a table or a plot underneath.

In **Google Colab**, click the triangle beside a code cell or press **Shift + Enter** to run it. In Jupyter, the same keyboard shortcut works. The variable names you create are kept in *kernel memory* until the runtime restarts. That means cells must normally be run **in order**.

### Tiny Python glossary

| Syntax | Meaning | Example |
|---|---|---|
| `x = 5` | Assign the value 5 to the name `x` | The `=` here is *assignment*, not a test of equality. |
| `print(x)` | Show something on screen | Parentheses pass an argument to a function. |
| `"stroke"` | A string: text between quotation marks | Column names are strings. |
| `["age", "bmi"]` | A list containing two strings | Lists keep items in order. |
| `df["age"]` | Select one column of a table called `df` | Returns a pandas Series. |
| `df[["age", "bmi"]]` | Select two columns | Returns a pandas DataFrame. |
| `.head(10)` | Method call: show the first ten rows | Periods access methods and attributes. |
| `# comment` | Human explanation ignored by Python | Leave good comments in code. |
| `True`, `False` | Boolean truth values | Useful in assertions and filtering. |
| `0.20` | A decimal number (20%) | Often used for a test-set fraction. |

**How a machine-learning table looks:** each **row** represents one observed person (one sample); each **column** is a variable. The **features** (`X`) are the inputs the model sees; the **target/label** (`y`) is what it is trained to predict. Here `stroke = 1` indicates a recorded stroke and `stroke = 0` indicates no recorded stroke.

### The big picture

`CSV → inspect → visualize → split → learn preprocessing on training only → train models → test on untouched data → explain models → question results`

A model is *not* given the answer (`stroke`) among its input features. During training, it sees the answer **separately** so it can learn relationships; during testing, it predicts without seeing the answer first.

## What was corrected from the Part 3 slides?

The PDF is the **source for the teaching goals**, not a guarantee that every screenshot represents best practice.

| Part 3 slide pattern | Better approach here | Why |
|---|---|---|
| Positional slicing (`iloc[:, :-1]`) and manually specified encoded columns (pp. 24, 29) | Select target/features by **column name** and retain original EBM feature names | Prevents accidental identifier/target inclusion and mislabeled explanations. |
| Plain SMOTE after one-hot encoding (pp. 26–28) | **SMOTENC** before one-hot encoding, using categorical-column names | Synthetic one-hot fractions can represent impossible categories. |
| Imputation/scaling shown without fully explicit split discipline | Fit these statistics on **training data only** | Prevents test information leaking into training. |
| Target plotted as a histogram (p. 22) | Binary target plotted as a **count bar chart** | Categories are clearer as discrete counts. |
| Evaluation emphasis on predictions instead of minority-class errors | Inspect **recall, precision, F1, PR-AUC, confusion matrix** and calibration warnings | A model that always predicts “no stroke” scores well on raw accuracy. |
| Percentages described as a patient's actual "stroke risk" (p. 30) | Say **model-estimated probability of label 1** | The model has not been clinically validated or calibrated for personal risk. |

**Why not just repeat the screenshots?** The goal is to **understand and reproduce the science**, while making genuine fixes transparent. Consequently the numeric results will not necessarily match the PDF screenshots. Official API references appear at the end.

---

# MODULE 1 · Setup and reading data

**Goal:** Know what packages, files, DataFrames, columns and missing values are. This corresponds to lecture pp. 10–20.

## Step 01 — Install missing packages

**Why:** Python comes with a standard library, but specialized tools such as InterpretML (`interpret`) and SMOTENC (`imbalanced-learn`, imported as `imblearn`) may not already be installed.

**What the cell does, in order:** `importlib.util.find_spec` checks whether an import name exists; a dictionary maps **import name → pip package name**; `missing` collects only absent packages; `subprocess.check_call` runs `python -m pip install` if needed. Installing requires internet access and permission to install packages. No install happens if both are present.

**Common issue:** If imports fail immediately after installation, use **Runtime → Restart session** in Colab and rerun from the top. Avoid running this installation step halfway through an otherwise active notebook.

**Lecture link:** pp. 10–14

In [ ]:
# Install only missing packages. Works in Colab and standard Python notebooks with pip access.
import importlib.util
import subprocess
import sys

requirements = {"interpret": "interpret", "imblearn": "imbalanced-learn"}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("InterpretML and imbalanced-learn are already installed.")

### Reading the result — Step 01

Expect either **"already installed"** or a short installation message. This does **not** train a model, open the CSV, or access private accounts. If installation fails because the environment is offline, the later InterpretML sections cannot run until the dependency is made available.

## Step 02 — Import your Python tools

**Why:** An `import` makes functions available without typing their full implementation. Think of each library as a toolbox.

- `pandas` (`pd`) is for spreadsheets/data tables. `numpy` (`np`) is for numeric arrays. `matplotlib.pyplot` (`plt`) draws charts.
- `sklearn` provides train/test splitting, preprocessing, logistic regression and metrics.
- `imblearn.over_sampling.SMOTENC` creates synthetic training examples for mixed data.
- `interpret.glassbox.ExplainableBoostingClassifier` is the inherently interpretable model used in the PDF.
- `RANDOM_STATE = 42` fixes pseudo-random operations so a second run is reproducible. It is **not** a quality score.

**Read the syntax:** `from module import name` imports a particular tool; `as pd` creates a shorter alias. `plt.rcParams.update(...)` changes *plot appearance*, not any observations or model behavior.

**Lecture link:** pp. 14, 17

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, RocCurveDisplay,
)
from imblearn.over_sampling import SMOTENC
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.18})
print("pandas:", pd.__version__)

### Reading the result — Step 02

The final `print` should show a pandas version. Nothing is fit in this step. If you see `ModuleNotFoundError`, revisit the first code cell. Later code relies on the names introduced here: executing it before this step will trigger `NameError`.

## Step 03 — Find and load the actual CSV file

**Why:** A model cannot learn from a file it has not loaded. `pd.read_csv(path)` reads comma-separated text into a pandas **DataFrame**; we name it `strokeDF` to match the PDF.

**How the cell works:** `possible_paths` lists likely locations (your notebook folder, `/content` in Colab, and this environment). `next(...)` picks the first existing file. If no file is found and we are in Colab, `files.upload()` asks you to choose the CSV. `assert` verifies that two expected columns (`id`, `stroke`) exist. `shape` reports `(rows, columns)`; `.head(10)` displays ten *examples*, not the entire dataset.

**If using the ZIP package:** extract both `.ipynb` and `.csv` into one folder. If using Colab, upload the CSV when prompted. **Never upload the PDF in place of the CSV.**

**Lecture link:** pp. 16, 18

In [ ]:
# Search standard paths, and prompt for upload in Google Colab only when needed.
DATASET_NAME = "healthcare-dataset-stroke-data.csv"
possible_paths = [
    Path(DATASET_NAME),
    Path.cwd() / DATASET_NAME,
    Path("/content") / DATASET_NAME,
    Path("/mnt/data") / DATASET_NAME,
]
dataset_path = next((p for p in possible_paths if p.is_file()), None)

if dataset_path is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Place {DATASET_NAME} next to the notebook, then rerun this cell."
        ) from exc
    print(f"Please upload {DATASET_NAME}.")
    uploaded = files.upload()
    if DATASET_NAME not in uploaded:
        raise FileNotFoundError(f"The upload must be named {DATASET_NAME}.")
    dataset_path = Path(DATASET_NAME)

strokeDF = pd.read_csv(dataset_path)
assert {"stroke", "id"}.issubset(strokeDF.columns), "Unexpected CSV schema."
print("Loaded:", dataset_path)
print("Shape:", strokeDF.shape)
display(strokeDF.head(10))

### Reading the result — Step 03

For the provided file expect **5,110 rows × 12 columns**. These are *observations*, not necessarily a representative sample of a population. A record may have a unique `id`, but `id` is not a medically meaningful feature. A sample preview lets you spot column names and values such as `Male`, `Yes`, `Private`, `formerly smoked`.

## Step 04 — Examine schema, statistics and missing data

**Why:** Before training, learn whether columns are numeric or categorical, whether values are missing, and whether a number looks implausible.

- `.tail(10)` displays the final ten rows; it checks you are not looking only at the beginning.
- `.info()` reports each column's **data type** (`int64`, `float64`, `object`) and how many entries are not missing.
- `.describe()` reports count/mean/standard deviation/percentiles for numeric columns. `50%` is the **median**, not a probability of stroke.
- `.isna().sum()` counts missing entries; `.duplicated().sum()` checks exact duplicates and repeated IDs.

**Terminology:** `NaN` stands for “not a number” and is used as a missing-value marker. **Missing does not mean zero**. A `0` in `hypertension` is a meaningful negative indicator; a missing BMI cannot be replaced with 0 without justification.

**Lecture link:** pp. 18–20

In [ ]:
print("Last 10 rows:")
display(strokeDF.tail(10))
print("\nDataFrame info:")
strokeDF.info()
print("\nNumerical summary:")
display(strokeDF.describe().T)
print("\nMissing values per column:")
display(strokeDF.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nDuplicate patient IDs:", int(strokeDF["id"].duplicated().sum()))
print("Duplicate complete rows:", int(strokeDF.duplicated().sum()))

### Reading the result — Step 04

In the supplied file, **BMI is missing for 201 records**. The column `smoking_status` includes the text `Unknown`, which is a *recorded category*, not automatically a pandas missing value. You should see numeric minimum/maximum values and checks for duplicates. An unusual maximum or skew does not automatically prove a data-entry error: investigate before dropping data.

## Step 05 — Practice pandas filtering and selecting columns

**Why (beginner extension):** Most data-science errors start as confusion about rows, columns or indexing.

`df['column']` is one column; `df[['a','b']]` is a two-column table; boolean expressions (`df['stroke'] == 1`) find rows matching a condition. `.loc` uses **row labels and column names**, whereas `.iloc` uses integer positions. Column names are safer than guessing where a column falls.

**Lecture link:** beginner extension to pp. 18–20

In [ ]:
# A mini-exercise in navigating data; none of these actions modify strokeDF.
print("Column names:", strokeDF.columns.tolist())
print("First three ages:", strokeDF["age"].head(3).tolist())
print("Two-column preview:")
display(strokeDF[["age", "bmi"]].head(5))
print("How many recorded stroke cases?", int((strokeDF["stroke"] == 1).sum()))
print("Two rows with stroke=1 (for inspection only):")
display(strokeDF.loc[strokeDF["stroke"] == 1, ["age", "avg_glucose_level", "stroke"]].head(2))

### Reading the result — Step 05

The stroke count should be **249**. The last preview is not training or prediction—it merely displays selected recorded outcomes. Notice how `.loc` chooses *both* rows and columns by meaning rather than position. Try substituting `"age"` with `"bmi"`, then rerun; the output changes, but the original dataset remains intact.

### Knowledge check: Tables and variables

1. What is the difference between a **row** and a **column**?
2. What do `X` and `y` mean in supervised machine learning?
3. Is the word `Unknown` automatically a missing value?
4. Why not fill missing BMI with zero?

<details><summary><b>Click to reveal the explanation and answers</b></summary>

1. A row is one observed record/person; a column is a variable shared by records.
2. `X` contains predictors/features; `y` contains the target/known answer.
3. No. It is a string category unless explicitly mapped to missing.
4. Zero is not a plausible typical BMI; imputing zero would invent systematically misleading measurements.

</details>

---

# MODULE 2 · Exploratory Data Analysis (EDA)

**Goal:** Read plots correctly and understand why class imbalance matters. PDF pp. 3–9 and 21–23. EDA is descriptive: it tells us what is in these data, not what causes stroke.

## Step 06 — Count positives and see the class imbalance

**Why:** Classification labels may not occur equally often. If ~95% of records are no-stroke, predicting “no stroke” for everyone looks accurate but utterly fails to detect stroke cases.

- `.value_counts()` counts each class.
- `.value_counts(normalize=True)` returns each class's **proportion** (adds to 1).
- Multiplying by `100` turns a proportion into a percentage.
- `.plot(kind='bar')` draws **discrete counts**; each bar represents a category.

**New term — prevalence:** In *this sampled dataset*, positive prevalence is `number of 1s ÷ number of all records`. Do **not** treat this as a regional or national clinical stroke prevalence.

**Lecture link:** pp. 3–6, 23

In [ ]:
class_counts = strokeDF["stroke"].value_counts().sort_index()
class_percent = strokeDF["stroke"].value_counts(normalize=True).sort_index().mul(100)
class_table = pd.DataFrame({"count": class_counts, "percent": class_percent.round(2)})
class_table.index = ["No stroke (0)", "Stroke (1)"]
display(class_table)

ax = class_counts.plot(kind="bar", rot=0, color=["#326e95", "#dd8052"])
ax.set_title("Target distribution: stroke vs. no stroke")
ax.set_ylabel("Number of patients")
ax.set_xticklabels(["No stroke (0)", "Stroke (1)"])
for i, count in enumerate(class_counts):
    ax.text(i, count + 30, f"{count:,}", ha="center")
plt.tight_layout()
plt.show()

print(f"Minority prevalence: {100 * strokeDF['stroke'].mean():.2f}%")
print("A model predicting 'no stroke' for every record can appear highly accurate,"
      " while having 0% stroke recall.")

### Reading the result — Step 06

Expect **4,861 no-stroke** and **249 stroke** observations—approximately **95.13% vs 4.87%**. That imbalance motivates careful evaluation, but imbalance is not always solved by automatically fabricating more cases. Your analysis goal (predict or explain) determines whether resampling helps.

## Step 07 — Prove why raw accuracy can fool you

**Why (critical extension):** Accuracy answers `correct predictions ÷ all observations`. A majority-class rule that *always* says 0 gets almost every record right, but it identifies **zero** stroke cases. This is a deliberately bad model to show the failure mode.

The following model has no training process: `np.zeros` creates an all-zero prediction vector. `recall_score(..., pos_label=1)` asks: "Of the actual stroke records, what fraction did we catch?"

**Lecture link:** extension: motivating imbalance on pp. 4–6

In [ ]:
# An intentionally useless classifier: always predict "no stroke".
majority_guess = np.zeros(len(strokeDF), dtype=int)
print("Accuracy of always guessing no stroke:",
      f"{accuracy_score(strokeDF['stroke'], majority_guess):.2%}")
print("Recall for recorded stroke:",
      f"{recall_score(strokeDF['stroke'], majority_guess, zero_division=0):.2%}")
print("Confusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(strokeDF["stroke"], majority_guess, labels=[0, 1]))

### Reading the result — Step 07

You should see **~95.13% accuracy** but **0.00% stroke recall**. The confusion matrix's bottom-left cell contains the missed stroke records (false negatives). This is why later comparisons include precision, recall, F1 and precision–recall curves. The example uses the full dataset solely to illustrate a metric, **not** to claim evaluated model performance.

## Step 08 — Read continuous-variable histograms

**Why:** A histogram groups a **continuous** measurement into intervals called *bins*. It shows where observations concentrate and whether values are skewed or have extreme tails.

The PDF illustrates histograms for `age`, `bmi`, and `avg_glucose_level`. Those columns have meaningful numeric distances: a five-year age difference is numerically meaningful; an arbitrary one-step difference between `work_type` categories is not.

**How:** `dropna()` skips missing numeric values; `bins=30` makes 30 intervals; `plt` draws three charts. `skew()` estimates numerical asymmetry: positive values imply a longer right tail; negative values imply a longer left tail. **Skew ≠ error.**

**Lecture link:** pp. 7–9, 21–22

In [ ]:
continuous_features = ["age", "avg_glucose_level", "bmi"]
binary_features = ["hypertension", "heart_disease"]
category_features = [
    "gender", "ever_married", "work_type", "Residence_type", "smoking_status"
]
categorical_features = category_features + binary_features  # Both binary flags are categorical.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, continuous_features):
    ax.hist(strokeDF[column].dropna(), bins=30, color="#447da4", edgecolor="white")
    ax.set_title(f"Histogram: {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Mean age:", round(strokeDF["age"].mean(), 2))
print("Continuous feature skewness:")
display(strokeDF[continuous_features].skew().round(3).to_frame("skewness"))
print("Histograms show patterns and possible outliers; they do not by themselves prove a data error.")

### Reading the result — Step 08

Ask of each plot: Where are most observations? Is the shape symmetrical? Are there multiple peaks? Could extreme observations deserve review? What proportion is missing? The computed mean age describes the sample, not the age of an “average stroke patient.” The histograms mix stroke and no-stroke records; they **do not** show causation or predictive ability by themselves.

## Step 09 — Plot binary outcomes and inspect category counts

**Why:** Binary variables and categories are better depicted by **bars** representing exact categories. A histogram may disguise discrete group meaning.

The two health flags (`hypertension` and `heart_disease`) are 0/1 fields. For `smoking_status`, the code groups records by the recorded label and shows **how often a stroke label appears in each group**. `groupby(...).agg(['count','mean'])` returns a count and mean; for 0/1 labels, the mean equals the fraction of 1s.

**Warning:** Group differences can reflect age, selection effects, missing information, confounding and other factors. They do **not** mean that smoking status alone caused an observed difference.

**Lecture link:** pp. 21–23

In [ ]:
# Discrete/binary features are better represented by bar charts than histograms.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column in zip(axes, ["hypertension", "heart_disease", "stroke"]):
    counts = strokeDF[column].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color="#5585a1")
    ax.set_title(f"Counts: {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Records")
plt.tight_layout()
plt.show()

print("Observed stroke fraction by smoking_status (descriptive, not causal):")
display(strokeDF.groupby("smoking_status", dropna=False)["stroke"]
        .agg(["count", "mean"]).rename(columns={"mean": "stroke_fraction"})
        .sort_values("stroke_fraction", ascending=False))
print("'Unknown' is an observed smoking-status label, not a missing (NaN) value.")

### Reading the result — Step 09

You should see one bar per class and a smoking-status summary table. The `mean` column is a **group-specific observed fraction in this dataset**, not a personal probability or causal risk ratio. Some subgroups can be small; always read the `count` alongside the `mean`.

## Step 10 — Check missingness by feature and by outcome

**Why (extension):** We know BMI is missing, but is missingness clustered in one class? Before filling values, inspect whether the percentage missing differs between recorded stroke and no-stroke rows.

A missingness pattern can signal data-collection bias. Even if the percentages differ, this test alone cannot tell us *why*. We will later fit a replacement value **on the training set only** so that no test-data information leaks into training.

**Lecture link:** extension to pp. 18–20

In [ ]:
print("Missing values per variable:")
display(strokeDF.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("BMI missingness by recorded label:")
display(strokeDF.assign(bmi_missing=strokeDF["bmi"].isna())
        .groupby("stroke")["bmi_missing"]
        .agg(count="size", missing="sum", missing_fraction="mean"))

### Reading the result — Step 10

The table separates **how many records** are in a group from **how many lack BMI**. The fraction is between 0 and 1. This is not proof that BMI is absent at random; it is an exploratory signal requiring a data-collection investigation.

### Knowledge check: EDA and imbalance

1. Why can a model be >95% accurate and still miss every stroke case?
2. What is the difference between a **bar chart** and a **histogram**?
3. If the distribution of glucose is right-skewed, does that prove bad measurements?
4. Does a higher observed stroke fraction in a category establish causality?

<details><summary><b>Click to reveal the explanation and answers</b></summary>

1. The majority class dominates the denominator; a trivial majority guess works for accuracy but has 0% positive recall.
2. Bar chart: distinct categories. Histogram: numeric values grouped into intervals or bins.
3. No. Right skew may be a legitimate property of the measured sample.
4. No. Correlation/descriptive association can be confounded and noncausal.

</details>

---

# MODULE 3 · Train/test splitting and leakage-safe baselines

**Goal:** Understand how to measure performance on unseen records. A split creates a **training sample** and a **held-out test sample**. The test labels must not be used to tune the model. This module deepens PDF pp. 23–27.

## Step 11 — Separate predictors and labels, then split

**Why:** `stroke` is the correct answer, not a feature the model should see. `id` is a record identifier, not a clinical predictor. We explicitly drop both from `X`.

- `.copy()` creates a separate object and avoids unintentional edits to the original DataFrame.
- `test_size=0.20` withholds 20% of records for final evaluation.
- `stratify=y` maintains a similar fraction of stroke cases in each partition (important with rare positives).
- `random_state=42` makes the split repeatable.
- The assertion checks that a record isn't simultaneously in both partitions.

**Crucial principle:** `train_test_split()` comes **before** fitting imputers, scalers, feature encoders or samplers. Otherwise information from the test set can leak into training and inflate reported performance.

**Lecture link:** pp. 23–25; corrected splitting methodology

In [ ]:
X = strokeDF.drop(columns=["id", "stroke"]).copy()
y = strokeDF["stroke"].astype(int).copy()
assert set(continuous_features + categorical_features) == set(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Training class counts:\n", y_train.value_counts().sort_index())
print("Testing class counts:\n", y_test.value_counts().sort_index())
print("Train prevalence:", round(y_train.mean() * 100, 2), "%")
print("Test prevalence:", round(y_test.mean() * 100, 2), "%")
assert set(X_train.index).isdisjoint(set(X_test.index))

### Reading the result — Step 11

Expect about **4,088 training records** and **1,022 test records**, with the stroke fraction in each near **4.87%**. Train and test feature tables each have **10 columns**; the original table had 12 because `id` and `stroke` were excluded. `y_train`/`y_test` are one-dimensional label vectors.

## Step 12 — Build and train a logistic regression baseline

**Why:** Logistic regression is a well-known interpretable model for binary classification. It produces an estimate between 0 and 1 using a weighted sum of feature values and a **sigmoid** transformation.

**Pipeline:** A **pipeline** is an ordered workflow that learns each necessary transformation from training data whenever you call `.fit()`.

- **Numeric columns:** `SimpleImputer(strategy='median')` fills missing values using *training* medians; `StandardScaler` recenters/rescales variables.
- **Categorical columns:** most-frequent imputation; then `OneHotEncoder(handle_unknown='ignore')` makes a separate 0/1 indicator for each category (necessary for linear regression).
- **Binary health flags:** `passthrough` keeps existing 0/1 values.
- **Classifier:** `LogisticRegression(class_weight='balanced')` gives the rarer class more training weight. Balanced training **does not guarantee calibrated probabilities** or high recall.

**Vocabulary:** `.fit(X_train, y_train)` learns the model; `.predict(X_test)` returns class decisions; `.predict_proba(X_test)` returns **model outputs**, not clinically validated risk.

**Lecture link:** pp. 17, 26–27; corrected leakage-safe baseline

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numerical", numerical_pipeline, continuous_features),
    ("categorical", categorical_pipeline, category_features),
    ("binary", "passthrough", binary_features),
], remainder="drop")

LR_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
    )),
])
LR_pipeline.fit(X_train, y_train)
print("Logistic regression trained. Example predictions:", LR_pipeline.predict(X_test.iloc[:10]))

### Reading the result — Step 12

If training succeeds, the output shows ten example predicted labels. **Do not decide whether the model is good from these ten examples.** We evaluate all test observations later. The pipeline bundles preprocessing and prediction so the exact same fitted transformations are used at inference time.

## Step 13 — See how predicted classes depend on probabilities

**Why (extension):** The decision rule commonly uses a threshold `p ≥ 0.5 → 1`; otherwise it predicts 0. With weighted models, 0.5 may not be the most sensible operational threshold.

`predict_proba` gives one number for `class 0` and one for `class 1`; the two add to approximately 1 for each row. We show **only a handful** to understand the data structure, not choose a threshold from outcomes.

**Lecture link:** extension to pp. 27–30

In [ ]:
small_probs = LR_pipeline.predict_proba(X_test.iloc[:5])
small_display = pd.DataFrame(small_probs, columns=["P(label=0)", "P(label=1)"])
small_display["Predicted class at 0.50"] = (small_display["P(label=1)"] >= 0.5).astype(int)
display(small_display.round(4))
print("Probability row sums:", np.round(small_probs.sum(axis=1), 6))

### Reading the result — Step 13

Each row should sum to 1 (within floating-point rounding). **Prediction** (the yes/no class) and **probability estimate** (a continuous score) are different outputs. Do not call a probability a real-world patient risk just because it looks like a percentage.

---

# MODULE 4 · Why missing values and mixed-type SMOTE matter

**Goal:** See the difference between copying minority records, inventing synthetic numeric samples, and properly handling category features. PDFs pp. 4–6 and 26–28.

## Concept primer — under/over-sampling vs SMOTE vs SMOTENC

Imagine 95 blue circles and 5 orange circles. A learner might nearly always say “blue.”

| Method | Main idea | Cost |
|---|---|---|
| **Undersampling** | Remove some majority samples | Faster but throws information away. |
| **Random oversampling** | Duplicate some minority samples | Simple but can overfit duplicated cases. |
| **SMOTE** | Synthesize minority samples between nearby numerical observations | Can interpolate into unrealistic regions; not suited directly to raw nominal categories. |
| **SMOTENC** | Mixed **N**umerical/**C**ategorical variant | Preserves discrete category values better, but new rows may still be implausible and alter calibration. |

**Important distinction:** Resampling is performed on **training observations only**. In a real study it must also occur *inside each training fold* of cross-validation, never before splitting. Synthetic training records are **not additional real patients**.

**Why standardize before neighbor-based sampling?** Age might span 0–82 while glucose spans ~55–270. An unscaled distance calculation may treat one feature as much larger just because of units. We scale continuous variables before SMOTENC and return them to original units afterwards.

## Step 14 — Fill missing values without seeing test data

**Why:** Most models require complete numeric inputs. We calculate medians and modes **on X_train only**, then reuse them for X_test. Otherwise, even a seemingly innocent median computed from the test set is a small piece of leaked information.

`training_medians` is a Series: its keys are column names, its values are medians. `fillna(...)` replaces genuinely missing entries; it does not alter nonmissing readings. `training_modes` holds the most frequent observed category for each categorical feature.

**Note:** The logistic regression pipeline already does its own training-only imputation. Here we create explicitly cleaned named DataFrames **for InterpretML and SMOTENC**, where keeping real column names aids interpretation.

**Lecture link:** pp. 19, 26; corrected explicit imputation

In [ ]:
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

training_medians = X_train[continuous_features].median()
for feature in continuous_features:
    X_train_clean[feature] = X_train_clean[feature].fillna(training_medians[feature])
    X_test_clean[feature] = X_test_clean[feature].fillna(training_medians[feature])

training_modes = {name: X_train[name].mode(dropna=True).iloc[0]
                  for name in categorical_features}
for feature in categorical_features:
    X_train_clean[feature] = X_train_clean[feature].fillna(training_modes[feature])
    X_test_clean[feature] = X_test_clean[feature].fillna(training_modes[feature])

assert not X_train_clean.isna().any().any()
assert not X_test_clean.isna().any().any()
assert X_train_clean.columns.tolist() == X_test_clean.columns.tolist()
print("Training medians (used for both sets):")
display(training_medians.to_frame("training_median"))

### Reading the result — Step 14

You should see the training-set medians. Both cleaned tables must pass assertions that there are **no missing values** and that they have the **same column order**. `Unknown` smoking status remains a real label. The original `strokeDF` is unchanged; only copies were filled.

## Step 15 — Apply SMOTENC to training rows only

**Why:** The slide code combines one-hot encoding and ordinary SMOTE. That can make invalid synthetic values such as `gender_Male=0.35` and `gender_Female=0.65`. **SMOTENC** is designed for mixed numeric and categorical predictors.

**Key lines:**
1. `smote_scaler.fit_transform(X_train_clean[...])` learns the numeric scale from training data **alone**.
2. `SMOTENC(categorical_features=..., sampling_strategy=0.5)` says which variables are categories and sets the resampled minority/majority ratio to **1:2** (not necessarily 1:1).
3. `.fit_resample(...)` creates the synthetic training rows.
4. `.inverse_transform(...)` returns age/BMI/glucose to their original units for interpretability.
5. `X_test_clean` and `y_test` remain untouched. We explicitly check this.

**Important:** Even SMOTENC can make implausible combinations (for instance, unrealistic cross-feature associations). It does **not** create real clinical data. We will compare against training with the original class distribution instead of assuming resampling is beneficial.

**Lecture link:** pp. 4–6, 26–28; SMOTENC correction

In [ ]:
smote_scaler = StandardScaler()
X_train_scaled = X_train_clean.copy()
X_train_scaled.loc[:, continuous_features] = smote_scaler.fit_transform(
    X_train_clean[continuous_features]
)

smotenc = SMOTENC(
    categorical_features=categorical_features,
    sampling_strategy=0.5,
    k_neighbors=5,
    random_state=RANDOM_STATE,
)
X_train_smote_scaled, y_train_smote = smotenc.fit_resample(X_train_scaled, y_train)

X_train_smote = X_train_smote_scaled.copy()
X_train_smote.loc[:, continuous_features] = smote_scaler.inverse_transform(
    X_train_smote_scaled[continuous_features]
)
# Binary columns are categorical features; preserve integer display for InterpretML.
for feature in binary_features:
    X_train_smote[feature] = X_train_smote[feature].astype(int)

assert X_train_smote.columns.tolist() == X_train.columns.tolist()
assert not X_train_smote.isna().any().any()
assert len(X_test_clean) == len(X_test), "The test set must not be resampled."
print("Before SMOTENC (training only):\n", y_train.value_counts().sort_index())
print("After SMOTENC (training only):\n", pd.Series(y_train_smote).value_counts().sort_index())
print("Held-out test set (unchanged):\n", y_test.value_counts().sort_index())

### Reading the result — Step 15

Only the **training class counts** should change. `y_train_smote` has more 1s than `y_train`; the test set class counts remain exactly the same as before. Compare the counts to verify that your technique did not contaminate the held-out evaluation set. Changing `sampling_strategy` changes model training—not the source CSV or the test records.

## Step 16 — Audit the synthetic category values

**Why (extension):** Synthetic data deserve checks just like original data. One easy check is that categorical columns should not contain unexpected fraction-like values or entirely new labels.

The code compares each resampled categorical feature's unique observed values with the original training values, and prints whether SMOTENC produced any *new category labels*. It does not prove the new joint combinations are medically plausible, but it catches one common programming error.

**Lecture link:** extension to pp. 26–28

In [ ]:
checks = []
for column in categorical_features:
    old_categories = set(X_train_clean[column].dropna().astype(str).unique())
    new_categories = set(X_train_smote[column].dropna().astype(str).unique())
    unexpected = sorted(new_categories - old_categories)
    checks.append({"feature": column,
                   "original_unique": len(old_categories),
                   "synthetic_unique": len(new_categories),
                   "unexpected_labels": unexpected})
category_audit = pd.DataFrame(checks)
display(category_audit)
assert all(len(items) == 0 for items in category_audit["unexpected_labels"])
print("Category-label audit passed. Clinical plausibility is NOT proven.")

### Reading the result — Step 16

The **unexpected_labels** column should contain empty lists. If you see a number such as `0.42` presented as a nominal category where only `0` and `1` were allowed, something in the sampler/preprocessing order went wrong. This audit checks allowed labels, **not** whether synthetic rows reflect biologically possible patients.

### Knowledge check: Leakage and SMOTENC

1. Why must the train/test split occur **before** SMOTENC?
2. Why is ordinary SMOTE problematic on one-hot encoded category flags?
3. Are synthetic minority examples real patient records?
4. Does balancing the training set guarantee good precision or properly calibrated probabilities?

<details><summary><b>Click to reveal the explanation and answers</b></summary>

1. If test examples influence synthetic training examples, the “unseen” test is contaminated and performance can look artificially good.
2. Interpolation can produce nonbinary fractional dummy values.
3. No. They are artificial examples designed to alter learning.
4. No. Precision depends on false positives/prevalence; calibration can worsen after resampling.

</details>

---

# MODULE 5 · Train an Explainable Boosting Machine (EBM)

**Goal:** Understand *glass-box* interpretability and compare an EBM trained on the original data to one trained after SMOTENC. This implements the PDF's central InterpretML laboratory, pp. 10–14 and 28–30.

## Concept primer — what does EBM learn?

The PDF calls **ExplainableBoostingClassifier** a glass-box model. It is a **generalized additive model** learned with boosting. It gives a useful combination of flexible curves and interpretable feature contributions.

For binary classification, think of the model as

\[
\text{score}(x) = \beta_0 + f_{\mathrm{age}}(x_{\mathrm{age}}) + f_{\mathrm{bmi}}(x_{\mathrm{bmi}}) + \cdots
\]

and

\[
P(y=1\mid x)=\frac{1}{1+e^{-\text{score}(x)}}.
\]

Here `f_age(...)` is a **learned shape function**, not a single constant coefficient. When a term is positive, it *increases the stroke-class log-odds score* for that prediction; a negative term decreases it. A score change of +0.5 is **not** a +50 percentage-point change in probability.

With `interactions=0`, every effect is one feature at a time, making the first explanation easier to understand. Real systems may include **pairwise interactions**, but that complicates the interpretation; this is a deliberate teaching simplification.

**Glass-box vs post-hoc:** EBM produces intrinsic explanations from its own terms. LIME/SHAP approximate or attribute other models' behavior; the lecture PDF 2 discusses these distinctions. Interpretability is not identical to correctness or fairness.

## Step 17 — Fit original-data and SMOTENC EBMs

**Why two models?** We want to see whether resampling helps recall/precision without assuming it does. Using the *same held-out test data* supports a direct comparison.

`ebm_feature_types` tells InterpretML which columns are continuous and which are categories. `feature_names` retains human-readable names. `outer_bags=4` and `max_rounds=450` deliberately reduce runtime for the lab; they are **not** claimed to be optimal hyperparameters.

`.fit(X_train_clean, y_train)` trains an EBM using real training rows. `.fit(X_train_smote, y_train_smote)` trains another EBM after synthetic resampling. Neither model sees the test labels during fitting. **Do not interpret the EBM's model weights as physiological mechanisms.**

**Lecture link:** pp. 13–14, 28; corrected EBM design

In [ ]:
ebm_feature_types = [
    "nominal" if feature in categorical_features else "continuous"
    for feature in X_train.columns
]
ebm_settings = dict(
    feature_names=X_train.columns.tolist(),
    feature_types=ebm_feature_types,
    interactions=0,
    outer_bags=4,
    max_rounds=450,
    early_stopping_rounds=50,
    learning_rate=0.025,
    n_jobs=2,
    random_state=RANDOM_STATE,
)

ebm_original = ExplainableBoostingClassifier(**ebm_settings)
ebm_original.fit(X_train_clean, y_train)
print("EBM without resampling trained.")

ebm_smote = ExplainableBoostingClassifier(**ebm_settings)
ebm_smote.fit(X_train_smote, y_train_smote)
print("EBM with SMOTENC trained.")
print("EBM features:", ebm_smote.feature_names_in_)

### Reading the result — Step 17

You should see **two training-complete messages** and a list of original feature names. The models are different because they have different training class distributions. Training can take longer than the other cells; Colab with a modern Python runtime is recommended.

## Step 18 — Inspect learned EBM terms and the difference from logistic regression

**Why (advanced extension):** Having trained a model is not the same as knowing what its *components* mean. We check EBM's `term_names_` and count the number of learned additive terms. With `interactions=0`, every term should correspond to one input feature.

Think of logistic regression as using a straight-line relationship in the log-odds scale; EBM can learn a non-straight function per feature. Extra flexibility can help prediction but also adds ways to overfit, especially with rare positives.

**Lecture link:** extension to pp. 13–14

In [ ]:
print("Predictor columns:", X_train_clean.columns.tolist())
print("EBM learned terms:", ebm_original.term_names_)
print("Number of EBM terms:", len(ebm_original.term_names_))
print("Test prediction matrix shape:",
      ebm_original.predict_proba(X_test_clean.iloc[:3]).shape)
assert len(ebm_original.term_names_) == X_train_clean.shape[1]

### Reading the result — Step 18

The model should report **10 main-effect terms** (one for each predictor). `predict_proba` for three records should produce a `(3, 2)` matrix: three rows, and separate probabilities for label 0 and 1. If an interaction is later enabled, this simple `10 terms` assertion must be removed or adapted.

---

# MODULE 6 · Evaluate beyond accuracy

**Goal:** Calculate predictive performance without being misled by class imbalance. This section extends the lecture, which discusses imbalanced data but does not provide a complete evaluation framework.

## Metric primer — read this before the scores

A **confusion matrix** counts four outcomes for positive class `stroke=1`:

| | Predicted no stroke (0) | Predicted stroke (1) |
|---|---:|---:|
| **Actually no stroke (0)** | TN: true negative | FP: false positive |
| **Actually stroke (1)** | FN: false negative | TP: true positive |

- **Accuracy** = `(TP + TN) / (TP + TN + FP + FN)` — can hide missed rare events.
- **Recall / sensitivity for stroke** = `TP / (TP + FN)` — among actual stroke cases, how many detected?
- **Precision for stroke** = `TP / (TP + FP)` — among cases the model flagged, how many were actual stroke cases?
- **F1** = harmonic mean of precision and recall; useful but does not measure calibration.
- **Balanced accuracy** = average of recall for class 0 and recall for class 1.
- **ROC-AUC** = discrimination over thresholds, but can look favorable even when positives are rare.
- **Average precision (AP)** summarizes the precision–recall curve. Its no-skill reference is **roughly the test positive prevalence**, so AP is especially helpful with imbalanced labels. It is not necessarily the same numerical formula as trapezoidal PR-AUC.
- **Brier score** = mean of squared differences between estimated probabilities and actual 0/1 outcomes; lower is better, but it should be interpreted with the outcome prevalence and calibration plots.

**Do not choose a classification threshold by repeatedly testing on your held-out test set.** That would leak the answer back into model design. A separate validation split or cross-validation should be used for threshold selection.

## Step 19 — Compute out-of-sample metrics for all models

**Why:** A single number is never enough. We evaluate all models on the **same original, untouched test labels** so differences reflect training/model choices instead of different test populations.

In a loop, `predict_proba` returns class-1 scores; `(probability >= 0.50)` converts them to yes/no decisions; each metric is calculated and stored in `results_df`. `classification_report` gives class-specific results. We report the number predicted positive because very different alert rates can explain precision/recall differences.

**Interpretation checklist:** A high recall with low precision means many positives were flagged, but many may be false alarms. Low recall means missed recorded strokes. Neither pattern establishes clinical utility without external validation and decision-cost analysis.

**Lecture link:** evaluation extension to pp. 4–6, 27–30

In [ ]:
models_to_evaluate = {
    "Logistic regression (weighted)": (LR_pipeline, X_test),
    "EBM (original training data)": (ebm_original, X_test_clean),
    "EBM (SMOTENC training data)": (ebm_smote, X_test_clean),
}

results = []
predictions = {}
for model_name, (fitted_model, X_eval) in models_to_evaluate.items():
    probability = fitted_model.predict_proba(X_eval)[:, 1]
    prediction = (probability >= 0.50).astype(int)
    predictions[model_name] = (prediction, probability)
    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
        "precision_stroke": precision_score(y_test, prediction, zero_division=0),
        "recall_stroke": recall_score(y_test, prediction, zero_division=0),
        "f1_stroke": f1_score(y_test, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probability),
        "average_precision": average_precision_score(y_test, probability),
        "brier_score": brier_score_loss(y_test, probability),
        "predicted_stroke_count": int(prediction.sum()),
    })

results_df = pd.DataFrame(results).set_index("model")
display(results_df.round(4))
print(f"Positive-class prevalence in the test set: {y_test.mean():.4f}")
for name, (prediction, _) in predictions.items():
    print("\n===", name, "===")
    print(classification_report(y_test, prediction, labels=[0, 1],
                                target_names=["No stroke", "Stroke"], zero_division=0))

### Reading the result — Step 19

Read the rows **horizontally** (each model's trade-offs) and **vertically** (same metric across models). Compare **recall, precision and AP** before discussing accuracy. A model with higher recall may have a worse Brier score after oversampling; that is an informative result, not a mistake in the notebook.

## Step 20 — Read confusion matrices, precision–recall and ROC curves

**Why:** Metrics summarize behavior, but charts expose how errors occur and how thresholds change trade-offs.

- **Confusion matrix:** two actual classes on the rows, two predicted classes on the columns. **FN** is the *lower-left* cell when labels are `[0,1]`; **FP** is the *upper-right*.
- **Precision–recall curve:** shows how precision and stroke recall trade off as the threshold moves. The dashed line represents test prevalence as a rough no-skill precision baseline.
- **ROC curve:** plots true-positive rate against false-positive rate across thresholds.

**Do not compare curves from one dataset as proof of real-world performance.** They are a teaching held-out estimate that can vary across random samples.

**Lecture link:** evaluation extension

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (model_name, (prediction, _)) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, prediction, labels=[0, 1],
        display_labels=["No stroke", "Stroke"],
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(model_name, fontsize=10)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, (_, probability) in predictions.items():
    PrecisionRecallDisplay.from_predictions(y_test, probability, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(y_test, probability, name=name, ax=axes[1])
axes[0].axhline(y_test.mean(), linestyle="--", color="black", label="Prevalence baseline")
axes[0].legend(fontsize=8)
axes[0].set_title("Precision–recall curves")
axes[1].set_title("ROC curves")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Reading the result — Step 20

Ask: Which model produces fewer false negatives? At what cost in false positives? Is the precision–recall curve consistently stronger than the no-skill baseline? Does ROC-AUC agree with AP? No curve automatically tells you what operational threshold to use.

## Step 21 — Explore the meaning of a threshold — without tuning on the test set

**Why (advanced extension):** A probability model is not inherently a fixed yes/no decision rule. At threshold 0.25 it flags more samples than at 0.75. This **teaching demonstration** uses test labels to *illustrate* the consequences. **Do not select a deployed threshold from this table**, because that would be test-set tuning.

We inspect the original-data EBM, not the oversampled one, and explicitly mark this result as a demonstration.

**Lecture link:** extension to pp. 27–30

In [ ]:
# DEMONSTRATION ONLY — not a procedure for choosing the threshold.
prob_demo = predictions["EBM (original training data)"][1]
threshold_table = []
for cutoff in [0.20, 0.50, 0.80]:
    guess = (prob_demo >= cutoff).astype(int)
    threshold_table.append({
        "threshold": cutoff,
        "alerts": int(guess.sum()),
        "recall_stroke": recall_score(y_test, guess, zero_division=0),
        "precision_stroke": precision_score(y_test, guess, zero_division=0),
    })
display(pd.DataFrame(threshold_table).round(3))

### Reading the result — Step 21

Typically, lowering the threshold produces **more alerts** and **higher recall**, often sacrificing precision. The trade-off is not guaranteed to change monotonically for precision in every sample. For a valid operational choice, design a separate validation procedure based on clinical priorities *before evaluating the test set*.

### Knowledge check: Metrics

1. What does **FN** count?
2. A model flags 100 people; 20 have recorded stroke. What is its precision?
3. A dataset contains 30 actual stroke cases; the model flags 24 of those. What is recall?
4. Why report AP as well as ROC-AUC for rare positives?
5. Can you use the held-out test set to select your decision threshold and still call it an untouched test?

<details><summary><b>Click to reveal the explanation and answers</b></summary>

1. False negatives: actual positive stroke cases predicted negative.
2. `20/100 = 0.20 = 20%` precision.
3. `24/30 = 0.80 = 80%` recall.
4. AP emphasizes precision at varying recall and has an interpretable low-prevalence baseline.
5. No. That would reuse test labels to tune the decision rule. Use validation data.

</details>

---

# MODULE 7 · Explain the model globally

**Goal:** Read feature importance and explain how a feature's values change EBM predictions. This is **global XAI**: it describes the model's learned behavior *across many training examples*, not one person.

## Step 22 — Rank global EBM feature contributions

**Why:** EBM exposes intrinsic **term importance** (how much the term matters across a sample on average). `term_importances()` returns a value for each model term; `term_names_` provides its name. The code plots the ten highest terms for both EBMs.

**Interpretation:** Importance ranks **model reliance**, not clinical causation, treatment benefit or statistical significance. Correlated predictors can distribute importance between themselves; resampling can change that ranking. A feature with low importance is not automatically irrelevant to real stroke mechanisms.

**Lecture link:** pp. 28–29, plus global-XAI extension

In [ ]:
importance_rows = []
for model_name, model in [
    ("EBM original", ebm_original), ("EBM SMOTENC", ebm_smote)
]:
    for term_name, importance in zip(model.term_names_, model.term_importances()):
        importance_rows.append({
            "model": model_name, "term": term_name, "importance": float(importance)
        })
importance_df = pd.DataFrame(importance_rows)
for model_name in importance_df["model"].unique():
    top = importance_df.loc[importance_df["model"] == model_name].nlargest(10, "importance")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.barh(top["term"][::-1], top["importance"][::-1], color="#467f99")
    ax.set_title(f"Global term importance — {model_name}")
    ax.set_xlabel("Mean absolute additive-score contribution")
    plt.tight_layout()
    plt.show()

### Reading the result — Step 22

Look for which terms rank high in each EBM and whether the ordering changes after SMOTENC. Read the axis: **average magnitude of additive-score contribution**. It is **not a percentage**. The plot says “the model uses this feature,” not “the feature causes strokes.”

## Step 23 — Open the interactive InterpretML global dashboard

**Why:** Bar charts show importance rankings but not *which values* of a predictor raise or lower the model's score. The interactive `ebm.explain_global()` dashboard lets you select an individual feature and inspect its learned shape function.

**How to explore:** On the summary chart, select `age` or `avg_glucose_level`. On that feature's shape plot, identify where the **model's additive log-odds contribution** goes above or below zero. Sudden spikes on tiny categories can indicate sparse-data instability. These plots are **exact features of this EBM**, unlike approximate post-hoc explanations.

`show(...)` may not render in every headless environment; the `except` fallback tries to display a static visualization. In Colab/Jupyter, this usually works best.

**Lecture link:** pp. 28–29; InterpretML global explanation

In [ ]:
# Interactive dashboard (works best in Colab/Jupyter with a supported renderer).
ebm_global = ebm_smote.explain_global(name="EBM global explanation (SMOTENC)")
try:
    show(ebm_global)
except Exception as exc:
    print("Interactive dashboard unavailable:", type(exc).__name__, str(exc)[:200])
    display(ebm_global.visualize())

### Reading the result — Step 23

**How to describe a feature graph in words:** “In this fitted model, certain age bins contribute positively to the score for label 1, while other bins contribute negatively.” Avoid phrases such as “age causes stroke because the graph rises.” The trained curve can reflect the dataset's collection process as much as genuine population relationships.

---

# MODULE 8 · Explain one prediction locally

**Goal:** Reproduce the PDF's patient-level walkthrough, while preserving correct mathematical meaning and avoiding the dangerous label “medical risk.” **Local XAI** explains a specific prediction for a specific record.

## Step 24 — Select a person from the held-out test data

**Why:** Local explanation refers to **one exact row**. The dataset's DataFrame index is no longer consecutive after splitting, so `.iloc[[position]]` selects by position and preserves the **single-row DataFrame** structure. The corresponding label comes from `y_test.iloc[position]`.

The cell prints the person's feature values, the actual recorded label and the estimated label-1 probabilities from all three models. **These are model outputs, not measured stroke probabilities.**

**Experiment:** Change `PATIENT_POSITION = 0` to 1 or 2 and rerun this cell **and the following explanation cells** to inspect a different held-out record. Do **not** choose one person as proof of model performance.

**Lecture link:** p. 30; corrected patient example

In [ ]:
PATIENT_POSITION = 0  # Change to any integer from 0 to len(X_test)-1.
assert 0 <= PATIENT_POSITION < len(X_test_clean)
patient = X_test_clean.iloc[[PATIENT_POSITION]]
true_label = int(y_test.iloc[PATIENT_POSITION])

print("Selected patient features (identifier excluded):")
display(patient.T.rename(columns={patient.index[0]: "value"}))
print("Recorded label:", "Stroke (1)" if true_label else "No stroke (0)")
print("Model-estimated P(class=stroke), NOT a validated clinical risk:")
for model_name, (fitted_model, X_eval) in models_to_evaluate.items():
    sample = X_test.iloc[[PATIENT_POSITION]] if model_name.startswith("Logistic") else patient
    positive_probability = float(fitted_model.predict_proba(sample)[0, 1])
    print(f"  {model_name}: {positive_probability:.2%}")

### Reading the result — Step 24

The printed record includes the original ten predictors; there is **no `stroke` column inside the features**. The true label appears separately. Predictions across models may differ because their learning strategies and training class distributions differ. One local case does not tell you which model is best overall.

## Step 25 — Render an individual's local explanation

**Why:** A global chart says “what generally matters”; a local chart says “which terms pushed **this particular model score** up or down for this record.” `ebm_smote.explain_local(patient, true_label)` returns one explanation; `show` displays its term contributions.

The model's base score/intercept is the starting point. Each feature contributes a positive or negative **log-odds score**. Add these and apply the inverse-logit/sigmoid to recover the output probability.

**Important:** These term effects are model-specific associations, not proof that changing a patient's feature would change their real health outcome. Some features (age, prior illness) are not actionable in any straightforward sense.

**Lecture link:** p. 30; local XAI

In [ ]:
local_explanation = ebm_smote.explain_local(
    patient, y_test.iloc[[PATIENT_POSITION]], name="EBM local explanation: chosen test patient"
)
try:
    show(local_explanation)
except Exception as exc:
    print("Interactive local explanation unavailable:", type(exc).__name__, str(exc)[:200])
    display(local_explanation.visualize(0))

### Reading the result — Step 25

In the local plot, a **positive** contribution increases the log-odds of label 1 relative to the model baseline. A **negative** contribution decreases it. Log-odds are not the same as probability points or percentage changes in clinical risk. Compare this visualization with the patient's profile printed in the preceding step.

## Step 26 — Reconstruct the probability from feature contributions

**Why (advanced but essential):** We verify that the additive explanation really accounts for the EBM prediction. `eval_terms(patient)` returns **one score for each feature term** for the person. We sum them with the model `intercept_`, then apply `expit`, the **sigmoid** function:

\[
\sigma(z) = \frac{1}{1+e^{-z}}.
\]

For instance, if the log-odds score were 0, sigmoid would give 0.5. If the score were negative, the probability of label 1 would be below 0.5. The code checks whether reconstructed and directly predicted values agree numerically.

**Why it matters:** This is not a post-hoc guess of what the EBM is doing; it is its internal additive calculation. `np.isclose(...)` allows for tiny floating-point rounding differences.

**Lecture link:** p. 30; EBM mathematics extension

In [ ]:
# Signed additive log-odds contributions, one per model term.
local_scores = ebm_smote.eval_terms(patient)[0]
local_contributions = pd.DataFrame({
    "term": ebm_smote.term_names_,
    "log_odds_contribution": local_scores.astype(float),
}).sort_values("log_odds_contribution")
display(local_contributions)

fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(local_contributions["log_odds_contribution"] >= 0, "#c86b48", "#3e78a1")
ax.barh(local_contributions["term"], local_contributions["log_odds_contribution"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Contribution to stroke-class log-odds (not probability points)")
ax.set_title("Local EBM explanation for the selected patient")
plt.tight_layout()
plt.show()

# For a binary logistic-link EBM, additive terms plus intercept reconstruct the model probability.
from scipy.special import expit
intercept = float(np.ravel(ebm_smote.intercept_)[0])
probability_from_terms = float(expit(intercept + np.sum(local_scores)))
probability_from_model = float(ebm_smote.predict_proba(patient)[0, 1])
print(f"Reconstructed model probability: {probability_from_terms:.6f}")
print(f"predict_proba output:           {probability_from_model:.6f}")
assert np.isclose(probability_from_terms, probability_from_model, rtol=1e-3, atol=1e-4)

### Reading the result — Step 26

The reconstructed probability and `predict_proba` should match to rounding. In the bar chart, **rightward positive terms increase the stroke-class model score** and leftward negative terms reduce it. A term of +1 does **not** mean “+100% risk.” Explaining a prediction is a different task from checking whether it is correct.

---

# MODULE 9 · Advanced model criticism

**Goal:** Move beyond producing charts: audit weak points in the analysis. This is an advanced extension, not a required part of the Part 3 slide demonstration.

## Step 27 — Investigate model calibration, not just classification

**Why:** A classifier could rank cases correctly but give misleading probabilities. *Calibration* asks whether a group of records given scores near, say, 0.20 contains approximately 20% positives. With few positive cases, these bins are noisy; treat the plot as an exploratory diagnostic rather than proof of clinical risk accuracy.

`calibration_curve(..., strategy='quantile')` groups predicted probabilities into roughly equally populated bins and compares mean predicted score against observed positive fraction. An idealized calibrated model follows the diagonal, but calibration is sample-size dependent. Oversampling often changes the training class prior and can distort predicted probabilities.

**Lecture link:** advanced extension

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(7, 6))
for model_name in ["EBM (original training data)", "EBM (SMOTENC training data)"]:
    probs = predictions[model_name][1]
    observed, expected = calibration_curve(y_test, probs, n_bins=8, strategy="quantile")
    plt.plot(expected, observed, marker="o", label=model_name)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration (reference)")
plt.xlabel("Mean model-estimated probability in bin")
plt.ylabel("Observed stroke fraction in same bin")
plt.title("Exploratory calibration (small, imbalanced test sample)")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Reading the result — Step 27

A curve far from the diagonal suggests probability-calibration problems on this test sample, but even apparent closeness here is **not clinical validation**. Avoid quoting bin-level percentages as population stroke probabilities. If calibration matters in a future real project, use **separate validation/calibration data** and external evaluation; never calibrate on the test outcomes reported here.

## Step 28 — Check basic subgroup error rates and sample sizes

**Why:** Explainability can reveal patterns, but it does not automatically establish fairness. A useful first check is to compare **subgroup sizes and error rates** for a sensitive or contextual attribute. Here we examine `gender` and `Residence_type` only as a *methodological audit*, not a causal claim.

Small subgroups have unstable metrics, especially if no positive cases appear. The code reports sample counts and the number of positive cases, and leaves recall undefined if no positives exist. **This is not a full fairness audit**: it ignores confounding, historical measurement bias, proxy features, selection effects and uncertainties.

**Lecture link:** advanced extension

In [ ]:
subgroup_rows = []
for group_col in ["gender", "Residence_type"]:
    for group_value, ids in X_test.groupby(group_col).groups.items():
        labels = y_test.loc[list(ids)]
        positive_scores = pd.Series(predictions["EBM (original training data)"][1], index=X_test.index).loc[list(ids)]
        flagged = (positive_scores >= 0.50).astype(int)
        subgroup_rows.append({
            "group_feature": group_col,
            "group": str(group_value),
            "n_records": int(len(labels)),
            "n_strokes": int(labels.sum()),
            "positive_prediction_fraction": float(flagged.mean()),
            "recall_stroke": (float(recall_score(labels, flagged)) if labels.sum() > 0 else np.nan),
        })
subgroup_report = pd.DataFrame(subgroup_rows)
display(subgroup_report.sort_values(["group_feature", "group"]).round(3))
print("WARNING: tiny subgroups/rare events make estimates unreliable.")

### Reading the result — Step 28

First read `n_records` and `n_strokes`; if these are tiny, ignore dramatic-looking percentages. Differences could arise from sampling noise, label definitions, age distributions, systematic under-recording or real model disparities. This table is an **audit prompt**, not a certification of group fairness.

## Step 29 — Make a cautious what-if prediction

**Why (advanced extension):** A model can recompute predictions when you supply different feature values. That does **not** estimate a treatment effect: changing one cell in a DataFrame is not equivalent to changing a person's real physiology.

We make a **hypothetical copy** of the selected test record and change `avg_glucose_level` by a small arbitrary amount. This is useful for learning how to call the model, but must never be interpreted as “lowering glucose by N will reduce stroke probability by M.”

**Lecture link:** advanced extension

In [ ]:
what_if = patient.copy()
original_glucose = float(what_if["avg_glucose_level"].iloc[0])
what_if.loc[:, "avg_glucose_level"] = original_glucose + 10.0
p_before = float(ebm_original.predict_proba(patient)[0, 1])
p_after = float(ebm_original.predict_proba(what_if)[0, 1])
print(f"Hypothetical glucose input: {original_glucose:.2f} -> {original_glucose+10:.2f}")
print(f"Original EBM predicted P(label=1): {p_before:.3f}")
print(f"Hypothetical EBM predicted P(label=1): {p_after:.3f}")
print("This is model sensitivity, NOT a clinical intervention effect.")

### Reading the result — Step 29

The numbers may move a little, a lot or not at all (EBMs use learned bins/functions). **Do not infer a treatment recommendation, dose change, or causal effect.** A model can respond to altered inputs even when such a combination was rarely or never observed in training data.

### Knowledge check: XAI and limitations

1. What does a **global** EBM explanation tell you? What does a **local** explanation tell you?
2. Does a positive feature contribution mean a positive change in probability measured in **percentage points**?
3. Can global importance establish that a variable **causes** stroke?
4. If oversampling raises recall but reduces precision and worsens Brier score, is that necessarily a coding error?
5. What further evidence would be necessary before using this model in a real healthcare workflow?

<details><summary><b>Click to reveal the explanation and answers</b></summary>

1. Global: behavior across many records. Local: additive model contributions for one exact prediction.
2. No; for binary EBM the contributions here are on the log-odds scale.
3. No; explanations are about model reliance, not causal mechanisms.
4. No. Resampling changes the training distribution and may create such trade-offs.
5. At minimum: representative data, independent external validation, proper missingness/data-quality analysis, calibrated probabilities if probabilities are used, threshold selection on separate data, subgroup error analysis, clinical utility assessment, governance and expert oversight.

</details>

# Capstone — explain the whole workflow as if teaching someone else

Complete these tasks **without looking at the answer sections**:

1. **Data literacy:** State the number of rows, features, positive labels, and missing BMI measurements. Explain why `id` is excluded.
2. **EDA:** Describe two histogram shapes; do not confuse descriptive association with cause.
3. **Imbalance:** Show the accuracy and recall of the deliberately useless all-negative baseline.
4. **Validation:** Explain the order of splitting, fitting preprocessing, resampling training data, training and final testing. Explain why changing that order can leak information.
5. **Modeling:** Contrast logistic regression and EBM, and say why you can inspect EBM shape functions directly.
6. **Evaluation:** Choose one model for further *research*, using **three metrics** and a confusion matrix. State at least one trade-off and one limitation.
7. **XAI:** Explain one global term and one local contribution correctly on the **log-odds** scale.
8. **Methodology:** Name at least three things needed before anyone could responsibly interpret model scores as clinical risk.

### Optional mastery experiments

- **Experiment A:** Rerun with a different `RANDOM_STATE` and compare scores. Why do they change despite identical code? (*Sampling variation*.)
- **Experiment B:** Try `sampling_strategy=0.25` versus `0.50` (change **only training resampling**), then compare recall/AP/Brier. Avoid tuning these choices to the held-out test.
- **Experiment C:** Set `interactions=1` in the EBM, **remove the 10-term assertion**, retrain and inspect the extra term. Which explanations become more complex?
- **Experiment D:** Choose three different `PATIENT_POSITION` values, and articulate how their local contributions differ without turning model scores into medical claims.

### Research-grade next steps (not implemented or validated here)

Use repeated stratified cross-validation with training-fold-only preprocessing and resampling; reserve a final unseen test set; select thresholds and calibrate using separate validation data; compare against stronger baselines; assess imbalance-aware clinical utility; audit missingness, rare categories, sampling bias, subgroup error intervals, external generalization and privacy/ethical considerations. No single notebook can establish mastery without interpreting results and applying these skills repeatedly.

### References and what comes from where

- **Course source 3:** *Fundamentals of XAI* (30-slide PDF), pp. 3–9 (imbalance, sampling, histograms); pp. 10–14 (InterpretML); pp. 15–30 (stroke laboratory). **Main requested source.**
- **Course source 2:** *Fundamentals of XAI* (62-slide PDF), glass-box/black-box, intrinsic/post-hoc, global/local concepts.
- **Course source 1:** *An Introduction to XAI* (16-slide PDF), why explanations matter and why explanations do not prove correctness.
- **Official InterpretML EBM API:** https://interpret.ml/docs/python/api/ExplainableBoostingClassifier.html
- **Official InterpretML classification walkthrough:** https://interpret.ml/docs/python/examples/interpretable-classification.html
- **Official SMOTENC API:** https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html
- **sklearn classification metrics:** https://scikit-learn.org/stable/modules/model_evaluation.html

**End-of-course test:** Can you justify every preprocessing choice, recalculate precision and recall from a confusion matrix, and interpret one EBM local feature contribution *without* calling it causation or a percentage-point change in risk? If yes, you have the foundations needed to pursue a real XAI project.